# 01 - Bronze Ingest (CSV → Delta)

In [ ]:
# Widgets for paths
dbutils.widgets.text("raw_base", "abfss://raw@<storage-account>.dfs.core.windows.net/olist")
dbutils.widgets.text("curated_base", "abfss://curated@<storage-account>.dfs.core.windows.net/olist")
RAW = dbutils.widgets.get("raw_base").rstrip("/")
CUR = dbutils.widgets.get("curated_base").rstrip("/")

In [ ]:
# %run ./utils   # (Databricks) uncomment if utils.py is in the same repo/workspace

In [ ]:
from pyspark.sql.functions import to_timestamp, to_date, year, month

# Map of Olist CSV files → logical table names
sources = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "product_category_name_translation": "product_category_name_translation.csv"
}

for table, file in sources.items():
    df = (spark.read
            .option("header", True)
            .option("multiLine", True)
            .csv(f"{RAW}/" + file))

    # Minimal normalization for 'orders' (date partitions)
    if table == "orders":
        df = (df
            .withColumn("order_ts", to_timestamp("order_purchase_timestamp"))
            .withColumn("order_date", to_date("order_ts"))
            .withColumn("order_year", year("order_date"))
            .withColumn("order_month", month("order_date")))

        (df.write
          .format("delta")
          .mode("overwrite")
          .partitionBy("order_year","order_month")
          .save(f"{CUR}/bronze/{table}"))
    else:
        (df.write
          .format("delta")
          .mode("overwrite")
          .save(f"{CUR}/bronze/{table}"))

print("Bronze ingestion complete.")

In [ ]:
# Simple checks
orders_bz = spark.read.format("delta").load(f"{CUR}/bronze/orders")
cnt = orders_bz.count()
assert cnt > 0, "No orders ingested"
print("Orders bronze count:", cnt)